# Qwen OCR on Databricks — Minimal POC

Extract text from document images using Qwen2-VL + vLLM on a classic GPU cluster.

**Cluster**: DBR 15.4 LTS ML, single GPU node (T4 16 GB or A10G 24 GB)

**Important**: Run this notebook on a fresh cluster or restart Python before running.
Only one model should be loaded on the GPU at a time.

## 0. Clear GPU

Kill any leftover GPU processes from previous runs to reclaim VRAM.

In [0]:
import subprocess
result = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
    capture_output=True, text=True,
)
for pid in result.stdout.strip().split("\n"):
    pid = pid.strip()
    if pid:
        print(f"Killing GPU process {pid}")
        subprocess.run(["kill", "-9", pid])
print("GPU cleared")

## 1. Install Dependencies

In [0]:
%pip install "vllm==0.6.3.post1" "transformers==4.46.0" "qwen-vl-utils>=0.0.8" "Pillow>=10.0"
dbutils.library.restartPython()

## 2. Generate Sample Documents

No external files needed — we create test documents with PIL.

In [0]:
from PIL import Image, ImageDraw, ImageFont
import base64

def create_sample_invoice(path: str) -> str:
    img = Image.new("RGB", (700, 500), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 14)
        font_bold = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf", 18)
    except OSError:
        font = ImageFont.load_default()
        font_bold = font
    draw.text((30, 20), "INVOICE", fill="black", font=font_bold)
    lines = [
        "Invoice #:   2024-0847",
        "Date:        March 15, 2024",
        "Vendor:      Acme Analytics Corp",
        "Bill To:     GlobalCo Financial Services",
        "",
        "Description              Qty     Unit Price     Amount",
        "-------------------------------------------------------",
        "Data Pipeline Setup        1     $12,500.00   $12,500.00",
        "ML Model Training          3      $2,916.67    $8,750.00",
        "Cloud Compute (hrs)      240         $20.00    $4,800.00",
        "-------------------------------------------------------",
        "                              Subtotal:       $26,050.00",
        "                              Tax (8%):        $2,084.00",
        "                              TOTAL DUE:      $28,134.00",
        "",
        "Payment Terms: Net 30",
        "Due Date: April 14, 2024",
    ]
    y = 60
    for line in lines:
        draw.text((30, y), line, fill="black", font=font)
        y += 24
    img.save(path)
    return path

def create_sample_receipt(path: str) -> str:
    img = Image.new("RGB", (400, 350), "white")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 13)
    except OSError:
        font = ImageFont.load_default()
    lines = [
        "RECEIPT",
        "",
        "Store: TechMart Electronics",
        "Date: 2024-03-20 14:32",
        "Cashier: #47",
        "",
        "Laptop Stand      $49.99",
        "USB-C Hub         $34.99",
        "Monitor Cable     $12.99",
        "",
        "Subtotal:         $97.97",
        "Tax:               $7.84",
        "Total:           $105.81",
        "Paid: Visa **4921",
    ]
    y = 15
    for line in lines:
        draw.text((20, y), line, fill="black", font=font)
        y += 22
    img.save(path)
    return path

invoice_path = create_sample_invoice("/tmp/sample_invoice.png")
receipt_path = create_sample_receipt("/tmp/sample_receipt.png")
print(f"Created: {invoice_path}, {receipt_path}")

with open(invoice_path, "rb") as f:
    b64 = base64.b64encode(f.read()).decode()
displayHTML(f'<img src="data:image/png;base64,{b64}" style="border:1px solid #ccc" />')

## 3. Load Qwen2-VL with vLLM

In [0]:
import torch
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from PIL import Image

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

# Use the HF processor for correct prompt formatting
processor = AutoProcessor.from_pretrained(MODEL_ID)

# Detect GPU memory and adjust settings
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
is_t4 = gpu_mem_gb < 20  # T4 = 16 GB, A10G = 24 GB
print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.0f} GB)")

llm = LLM(
    model=MODEL_ID,
    dtype="half",
    max_model_len=2048 if is_t4 else 4096,
    gpu_memory_utilization=0.80,
    enforce_eager=True,
    limit_mm_per_prompt={"image": 1},
)
print("vLLM engine ready")

## 4. Run OCR — Single Document

In [0]:
from qwen_vl_utils import process_vision_info

def ocr_extract(image_path: str, prompt_text: str = "Extract all text and numbers from this document. Return the structured fields.") -> str:
    """Run OCR on a single image using vLLM."""
    messages = [{"role": "user", "content": [
        {"type": "image", "image": f"file://{image_path}"},
        {"type": "text", "text": prompt_text},
    ]}]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    img = Image.open(image_path).convert("RGB")
    outputs = llm.generate(
        [{"prompt": prompt, "multi_modal_data": {"image": img}}],
        sampling_params=SamplingParams(temperature=0.1, max_tokens=512),
    )
    return outputs[0].outputs[0].text

result = ocr_extract(invoice_path)
print("=== Invoice OCR ===")
print(result)

## 5. Batch OCR — Multiple Documents

In [0]:
docs = [
    {"path": invoice_path, "label": "Invoice"},
    {"path": receipt_path, "label": "Receipt"},
]

requests = []
for doc in docs:
    msg = [{"role": "user", "content": [
        {"type": "image", "image": f"file://{doc['path']}"},
        {"type": "text", "text": "Extract all text and numbers from this document. Return the structured fields."},
    ]}]
    p = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    img = Image.open(doc["path"]).convert("RGB")
    requests.append({"prompt": p, "multi_modal_data": {"image": img}})

batch_outputs = llm.generate(requests, sampling_params=SamplingParams(temperature=0.1, max_tokens=512))

for doc, out in zip(docs, batch_outputs):
    print(f"=== {doc['label']} ===")
    print(out.outputs[0].text)
    print()

## 6. Next Steps

This POC confirms Qwen2-VL OCR works on Databricks classic GPU with vLLM.

**From here you can:**
- Swap in your own document images (PDFs converted via `pdf2image`, scanned docs, etc.)
- Fine-tune with LoRA for your domain — see the companion notebook `vLLM_finetune_example`
- Register the model to Unity Catalog and deploy to Model Serving for production
- Use MLflow tracing to observe inference in production

**Model options:**
| Model | Params | VRAM (fp16) | Best for |
|---|---|---|---|
| `Qwen/Qwen2-VL-2B-Instruct` | 2B | ~4 GB | POC, light OCR, T4 nodes |
| `Qwen/Qwen2-VL-7B-Instruct` | 7B | ~14 GB | Production OCR, A10G nodes |
| `Qwen/Qwen2.5-VL-3B-Instruct` | 3B | ~6 GB | Newer model, improved accuracy |
| `Qwen/Qwen2.5-VL-7B-Instruct` | 7B | ~14 GB | Best quality, complex layouts |